# Análisis de stocks: métodos con 100% de hits en los últimos 5 años

Lee todos los `{Ticket}_valuaciones_hits_por_anio.xlsx` de `Valuaciones_USA/` (generados por
`Valuaciones_todas.ipynb`) y busca, por ticker y método de valuación, si en los últimos 5 años
el precio llegó a tocar la banda del objetivo en el 100% de los periodos.

Criterios usados (decididos con el usuario):
- Se piden al menos 5 periodos dentro de la ventana de 5 años para que el 100% cuente como señal.
- "Últimos 5 años" se cuenta hacia atrás desde el último periodo disponible **de cada ticker**
  (no desde la fecha de hoy), igual que el propio filtro de 10 años en `Valuaciones_todas.ipynb`.
- Si el ÚNICO periodo que falló (`hit=0`) dentro de la ventana es justo el último periodo (el
  más reciente), no cuenta en contra del hit rate: todavía puede no haberle dado tiempo de tocar
  la banda. Si el que falló es un periodo más viejo (no el último), sí cuenta como fallo real y
  el método no entra al 100%. La columna `Periodo_fallo` siempre muestra cuál fue el periodo que
  falló (y si ya se puede confirmar como fallo o sigue pendiente), aunque no haya bajado el
  Hit_rate_%.
- Nota: estos archivos ya vienen filtrados aguas arriba (solo incluyen métodos con hit rate
  >=70% en sus últimos 10 años). Un método con 100% en 5 años pero flojo entre el año 6 y el 10
  no va a aparecer aquí porque ni siquiera tiene fila en el archivo de origen.

In [ ]:
import glob
import os
import re

import numpy as np
import pandas as pd

CARPETA_VALUACIONES = "Valuaciones_USA"
PATRON_ARCHIVO = "_valuaciones_hits_por_anio.xlsx"
VENTANA_ANIOS = 5
MINIMO_PERIODOS = 5

## Cargar y apilar todos los archivos de hits por año

In [ ]:
def listar_archivos_hits(carpeta=CARPETA_VALUACIONES, patron=PATRON_ARCHIVO):
    """Rutas de los archivos {Ticket}_valuaciones_hits_por_anio.xlsx, sin los temporales ~$."""
    rutas = glob.glob(os.path.join(carpeta, f"*{patron}"))
    return sorted(r for r in rutas if not os.path.basename(r).startswith('~$'))


def cargar_hits(carpeta=CARPETA_VALUACIONES, patron=PATRON_ARCHIVO):
    """Concatena todos los archivos de hits por año en un solo DataFrame, con columna Ticket."""
    piezas = []
    for ruta in listar_archivos_hits(carpeta, patron):
        ticker = os.path.basename(ruta)[:-len(patron)]
        df = pd.read_excel(ruta)
        df.insert(0, 'Ticket', ticker)
        piezas.append(df)
    if not piezas:
        return pd.DataFrame(columns=['Ticket', 'Año', 'Metodo', 'Valor', 'hit', 'tth_weeks', 'hit_date', 'side'])
    return pd.concat(piezas, ignore_index=True)


hits = cargar_hits()
hits['Fecha'] = pd.to_datetime(hits['Año'], format='%b%Y') + pd.offsets.MonthEnd(0)
print(f"{hits['Ticket'].nunique()} tickers, {len(hits)} filas (Ticket x Metodo x Año)")

36 tickers, 2169 filas (Ticket x Metodo x Año)


C:\Users\mar_c\AppData\Local\Temp\ipykernel_24832\3566745275.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(piezas, ignore_index=True)
C:\Users\mar_c\AppData\Local\Temp\ipykernel_24832\3566745275.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(piezas, ignore_index=True)


## Ventana de 5 años por ticker y cálculo del hit rate por método

In [ ]:
def _filas_en_ventana(hits, anios=VENTANA_ANIOS):
    """Filas de `hits` dentro de los últimos `anios` años de cada ticker (desde su último
    periodo disponible), con `hit` no nulo y convertido a entero."""
    fecha_max_ticker = hits.groupby('Ticket')['Fecha'].transform('max')
    inicio_ventana = fecha_max_ticker - pd.DateOffset(years=anios)
    en_ventana = hits[(hits['Fecha'] >= inicio_ventana) & hits['hit'].notna()].copy()
    en_ventana['hit'] = en_ventana['hit'].astype(int)
    return en_ventana


def hit_rate_ultimos_n_anios(hits, anios=VENTANA_ANIOS, minimo=MINIMO_PERIODOS):
    """Hit rate por Ticket+Metodo en los últimos `anios` años (desde el último periodo de cada
    ticker), exigiendo al menos `minimo` periodos con hit no nulo dentro de la ventana.

    Si el ÚNICO periodo que falló (hit=0) es justo el último de la ventana, no cuenta en contra:
    puede que simplemente no le haya dado tiempo de tocar la banda todavía. Si el que falló es
    un periodo más viejo (no el último), sí cuenta como fallo real y baja el hit rate.
    """
    en_ventana = _filas_en_ventana(hits, anios)

    base = (
        en_ventana.groupby(['Ticket', 'Metodo'])
        .agg(
            N=('hit', 'size'),
            Hits=('hit', 'sum'),
            Primer_periodo=('Año', 'first'),
            Semanas_promedio_hasta_hit=('tth_weeks', 'mean'),
        )
        .reset_index()
    )

    idx_ultimo = en_ventana.groupby(['Ticket', 'Metodo'])['Fecha'].idxmax()
    ultimo = en_ventana.loc[idx_ultimo, ['Ticket', 'Metodo', 'Año', 'hit']].rename(
        columns={'Año': 'Ultimo_periodo', 'hit': 'Ultimo_hit'})

    resumen = base.merge(ultimo, on=['Ticket', 'Metodo'])
    # Si el último periodo de la ventana falló, no se cuenta en contra todavía.
    resumen['Hits_ajustado'] = resumen['Hits'] + (resumen['Ultimo_hit'] == 0).astype(int)
    resumen['Hit_rate_%'] = (resumen['Hits_ajustado'] / resumen['N'] * 100).round(1)
    resumen = resumen.drop(columns=['Ultimo_hit'])

    return resumen[resumen['N'] >= minimo].sort_values(['Ticket', 'Metodo']).reset_index(drop=True)


def metodos_con_umbral(resumen, umbral):
    """Filas de `resumen` (Ticket+Metodo) con Hit_rate_% >= umbral."""
    return resumen[resumen['Hit_rate_%'] >= umbral].sort_values(
        ['Ticket', 'Metodo']).reset_index(drop=True)


def con_periodo_de_fallo(resumen, detalle):
    """Agrega la columna Periodo_fallo: el (los) periodo(s) con hit=0 real (sin el ajuste del
    último periodo), marcando si ya se puede confirmar como fallo definitivo (pasaron 156
    semanas) o sigue pendiente. Se deja visible aunque el Hit_rate_% ya no lo cuente en contra
    cuando ese periodo es el último de la ventana."""
    fallos = detalle.loc[detalle['hit'] == 0, ['Ticket', 'Metodo', 'Año', 'Fecha']].copy()
    semanas_transcurridas = (pd.Timestamp.today() - fallos['Fecha']).dt.days / 7
    fallos['Periodo_fallo'] = np.where(
        semanas_transcurridas >= 156,
        fallos['Año'] + ' (confirmado)',
        fallos['Año'] + ' (pendiente)'
    )
    fallos = fallos.groupby(['Ticket', 'Metodo'])['Periodo_fallo'].agg(', '.join).reset_index()
    return resumen.merge(fallos, on=['Ticket', 'Metodo'], how='left').fillna({'Periodo_fallo': ''})


def con_conteo_hits_side(resumen, detalle):
    """Agrega Hits_long, Hits_short y Hits_at_target: de los hits (hit=1) dentro de la ventana,
    cuántos fueron con el precio viniendo de abajo del objetivo (long), de arriba (short), o ya
    estaban dentro de la banda desde el primer periodo de la ventana (at_target). Las tres
    columnas siempre suman Hits."""
    aciertos = detalle[detalle['hit'] == 1]
    conteo = (
        aciertos.groupby(['Ticket', 'Metodo', 'side']).size()
        .unstack(fill_value=0)
        .reindex(columns=['long', 'short', 'at_target'], fill_value=0)
        .rename(columns={'long': 'Hits_long', 'short': 'Hits_short', 'at_target': 'Hits_at_target'})
        .reset_index()
    )
    resumen = resumen.merge(conteo, on=['Ticket', 'Metodo'], how='left')
    for col in ('Hits_long', 'Hits_short', 'Hits_at_target'):
        resumen[col] = resumen[col].fillna(0).astype(int)
    return resumen


resumen_5y = hit_rate_ultimos_n_anios(hits)
detalle_5y = _filas_en_ventana(hits)   # mismas filas, pero sin agrupar: una por Ticket+Metodo+Año

metodos_100 = con_conteo_hits_side(
    con_periodo_de_fallo(metodos_con_umbral(resumen_5y, 100.0), detalle_5y),
    detalle_5y
)
print(f"Tickers con al menos un método al 100% en los últimos {VENTANA_ANIOS} años "
      f"(mínimo {MINIMO_PERIODOS} periodos; un fallo solo en el último periodo no cuenta en "
      f"contra): {metodos_100['Ticket'].nunique()}")
metodos_100

Tickers con al menos un método al 100% en los últimos 5 años (mínimo 5 periodos; un fallo solo en el último periodo no cuenta en contra): 28


,Ticket,Metodo,N,Hits,Primer_periodo,Semanas_promedio_hasta_hit,Ultimo_periodo,Hits_ajustado,Hit_rate_%,Periodo_fallo,Hits_long,Hits_short,Hits_at_target
0,ACN,Intrinsic_CAGR_FCF,6,5,Aug2021,37.800000,Nov2025,6,100.0,Nov2025 (pendiente),1,3,1
1,ACN,Intrinsic_FCF_Avg,6,5,Aug2021,17.200000,Nov2025,6,100.0,Nov2025 (pendiente),0,2,3
2,ACN,Intrinsic_Mediana,6,6,Aug2021,39.333333,Nov2025,6,100.0,,3,2,1
3,ADBE,Peter_Lynch_Value_2,6,6,Nov2019,6.000000,Aug2024,6,100.0,,0,3,3
4,ADP,GF_Value_Guru,6,6,Jun2021,6.000000,Sep2025,6,100.0,,2,1,3
5,ADSK,GF_Value_Guru,6,6,Jan2021,42.166667,Oct2025,6,100.0,,2,2,2
6,AMGN,GF_Value_Guru,6,6,Dec2020,15.666667,Sep2025,6,100.0,,5,0,1
7,AMGN,Median_PS_Value_Guru,6,6,Dec2020,16.000000,Sep2025,6,100.0,,5,0,1
8,AMZN,GF_Value_Guru,6,6,Dec2020,32.166667,Sep2025,6,100.0,,2,2,2
9,BAC,Book_Value_per_Share,6,5,Dec2020,6.800000,Sep2025,6,100.0,Sep2025 (pendiente),1,2,2


## Listado de tickets (únicos) con 100%

In [ ]:
tickers_100 = sorted(metodos_100['Ticket'].unique())
print(tickers_100)

['ACN', 'ADBE', 'ADP', 'ADSK', 'AMGN', 'AMZN', 'BAC', 'BRO', 'CDNS', 'CRM', 'DECK', 'FISV', 'FTNT', 'GOOG', 'HD', 'INTU', 'LMT', 'META', 'MMC', 'MNST', 'MORN', 'MSFT', 'NFLX', 'PEP', 'ROP', 'TSM', 'V', 'VRTX']


## Cuánto cambió el precio hasta llegar al objetivo en cada hit

Para cada hit (`hit=1`) de los métodos al 100% en la ventana de 5 años, se toma el precio más
temprano del mes del periodo (por ejemplo, para `Aug2021`, la apertura de la primera semana de
agosto de 2021, de `USA/DatosHistoricos/Datos históricos {Ticket}.csv`) y se compara contra el
campo `Valor` del archivo de valuaciones:

`Cambio_% = (Valor / Precio_inicio - 1) * 100`

- `long`: el precio venía de abajo, se espera un cambio positivo.
- `short`: el precio venía de arriba, se espera un cambio negativo.
- `at_target`: ya estaba dentro de la banda, se espera un cambio cercano a cero.
- Los precios de los CSV parecen ajustados por dividendos y `Valor` no, lo que puede distorsionar
  un poco los periodos más viejos.

In [ ]:
CARPETA_PRECIOS = os.path.join("USA", "DatosHistoricos")


def precio_inicio_por_mes(ticker, carpeta=CARPETA_PRECIOS):
    """Primera fila semanal de cada mes del histórico de precios del ticker.
    Devuelve Ticket, Mes (Period), Fecha_inicio y Precio_inicio (Apertura de esa semana)."""
    ruta = os.path.join(carpeta, f"Datos históricos {ticker}.csv")
    px = pd.read_csv(ruta, encoding='utf-8-sig', parse_dates=['Fecha']).sort_values('Fecha')
    px['Mes'] = px['Fecha'].dt.to_period('M')
    ini = px.groupby('Mes').first().reset_index()
    ini = ini.rename(columns={'Fecha': 'Fecha_inicio', 'Apertura': 'Precio_inicio'})
    ini.insert(0, 'Ticket', ticker)
    return ini[['Ticket', 'Mes', 'Fecha_inicio', 'Precio_inicio']]


# Hits (hit=1) dentro de la ventana de 5 años, solo de los métodos que quedaron al 100%.
cambio_hits = detalle_5y.merge(metodos_100[['Ticket', 'Metodo']], on=['Ticket', 'Metodo'])
cambio_hits = cambio_hits[cambio_hits['hit'] == 1].copy()
cambio_hits['Mes'] = cambio_hits['Fecha'].dt.to_period('M')

precios_inicio = pd.concat([precio_inicio_por_mes(t) for t in tickers_100], ignore_index=True)
cambio_hits = cambio_hits.merge(precios_inicio, on=['Ticket', 'Mes'], how='left')

# Cambio % desde el precio más temprano del mes del periodo hasta el Valor objetivo.
cambio_hits['Cambio_%'] = (cambio_hits['Valor'] / cambio_hits['Precio_inicio'] - 1) * 100

cambio_hits = cambio_hits[
    ['Ticket', 'Metodo', 'Año', 'Fecha_inicio', 'Precio_inicio', 'Valor',
     'Cambio_%', 'side', 'tth_weeks', 'hit_date']
].sort_values(['Ticket', 'Metodo', 'Fecha_inicio']).reset_index(drop=True)

print(f"{len(cambio_hits)} hits, {cambio_hits['Precio_inicio'].isna().sum()} sin precio de inicio")
cambio_hits

317 hits, 0 sin precio de inicio


,Ticket,Metodo,Año,Fecha_inicio,Precio_inicio,Valor,Cambio_%,side,tth_weeks,hit_date
0,ACN,Intrinsic_CAGR_FCF,Aug2021,2021-08-02,299.955952,311.424628,3.823453,at_target,0.0,2021-09-06
1,ACN,Intrinsic_CAGR_FCF,Aug2022,2022-08-01,287.682310,250.309261,-12.991084,short,1.0,2022-09-12
2,ACN,Intrinsic_CAGR_FCF,Aug2023,2023-08-07,303.510722,239.557763,-21.071071,short,100.0,2025-08-04
3,ACN,Intrinsic_CAGR_FCF,Aug2024,2024-08-05,305.978216,205.608437,-32.802916,short,75.0,2026-02-09
4,ACN,Intrinsic_CAGR_FCF,Aug2025,2025-08-04,258.079987,293.406801,13.688320,long,13.0,2025-12-01
...,...,...,...,...,...,...,...,...,...,...
312,VRTX,Median_ps_value_promedio_mediana,Dec2021,2021-12-06,203.729996,376.042589,84.578902,long,80.0,2023-07-17
313,VRTX,Median_ps_value_promedio_mediana,Dec2022,2022-12-05,318.850006,446.148193,39.924160,long,53.0,2024-01-08
314,VRTX,Median_ps_value_promedio_mediana,Dec2023,2023-12-04,349.000000,454.852970,30.330364,long,1.0,2024-01-08
315,VRTX,Median_ps_value_promedio_mediana,Dec2024,2024-12-02,467.480011,468.460844,0.209813,long,3.0,2025-01-27


In [ ]:
def resumen_cambio(df, por):
    """Estadísticos del Cambio_% agrupando por la(s) columna(s) `por`."""
    return (
        df.groupby(por)['Cambio_%']
        .agg(Hits='count', Promedio='mean', Mediana='median', Minimo='min', Maximo='max')
        .round(1)
        .reset_index()
    )

print("Por side (long = venía de abajo, short = venía de arriba, at_target = ya en la banda):")
display(resumen_cambio(cambio_hits, 'side'))


print("Por método:")
display(resumen_cambio(cambio_hits, 'Metodo').sort_values('Mediana', ascending=False))

cambio_hits.to_excel("Cambio_precio_hasta_hit.xlsx", index=False)
print(f"Guardado Cambio_precio_hasta_hit.xlsx ({len(cambio_hits)} filas)")

Por side (long = venía de abajo, short = venía de arriba, at_target = ya en la banda):


,side,Hits,Promedio,Mediana,Minimo,Maximo
0,at_target,107,3.4,2.1,-22.5,33.4
1,long,123,50.0,32.3,-0.7,344.4
2,short,87,-17.2,-17.6,-37.2,6.7


Por método:


,Metodo,Hits,Promedio,Mediana,Minimo,Maximo
12,Intrinsic_Weighted,6,64.9,62.6,-0.9,183.2
7,Intrinsic_EPS_Avg,11,36.7,38.4,-19.5,128.8
5,Intrinsic_CAGR_EPS,6,39.8,26.7,-17.4,165.9
15,Median_ps_value_promedio_mediana,17,27.8,24.3,-26.0,229.2
14,Median_ps_value_promedio,17,31.1,23.4,-25.6,255.0
11,Intrinsic_Mediana,11,32.5,15.4,-29.1,162.6
8,Intrinsic_EPS_Mediana,17,14.0,11.8,-26.1,97.0
16,Peter_Lynch_Value_1,6,12.1,11.3,-30.7,64.6
4,Graham_Number_Guru,6,13.2,9.7,-5.3,47.4
10,Intrinsic_FCF_Avg,10,35.8,9.0,-22.9,183.2


Guardado Cambio_precio_hasta_hit.xlsx (317 filas)


## Backtest: carteras anuales de 20,000 USD (solo compra, 12 meses fijos)

Para cada año se arma una cartera y se mide su retorno real a 12 meses. Los tickers y métodos
elegidos cada año se controlan con `FUENTE_CANDIDATOS`:

- **`'cambio_hits'` (por defecto):** los de la tabla `cambio_hits`, es decir, los (ticker, método,
  periodo) de los métodos al 100% en los últimos 5 años que llegaron a su objetivo (`hit=1`). El
  año de cada fila es el año de su periodo. Cubre 2020-2024 con muestra suficiente.
  **Ojo:** esta selección usa información futura (qué métodos terminaron en 100% y qué periodos
  pegaron). Sirve para ver cuánto movimiento tuvieron esos hits, pero **sobrestima** lo que se
  habría ganado decidiendo en tiempo real.
- **`'walk_forward'`:** los que pasaban el filtro de 100% de hits en el momento de decidir, usando
  solo periodos ya resueltos (aciertos con `hit_date` ocurrido y fallos confirmados a 156 semanas).
  No usa información futura.

Reglas comunes:
- **Selección:** solo se compra si `Valor` > precio de inicio del periodo. Se ordena por la
  mediana del cambio esperado de los métodos del ticker y se toman los N mejores.
- **Peso:** `igual`, o `consenso` (proporcional al número de métodos que respaldan al ticker).
- **Tiempo de compra:** en el mes del periodo o 3 / 6 meses después. Cada ticker compra en su propio
  mes de periodo. Retorno = apertura de la 1a semana del mes de compra → apertura 12 meses después.
- **Comparación:** S&P 500 Total Return con las mismas fechas y pesos, y carteras al azar con el
  mismo número de acciones (peso igual, mismas fechas) sacadas de los tickers que tenían periodo ese año.
- **Limitaciones:** (1) las 36 acciones de la muestra son empresas que sobrevivieron, por eso el azar
  también rinde bien; (2) los archivos de valuaciones ya vienen filtrados con hit rate >= 70% en 10
  años, lo que filtra con información futura; (3) se asume que `Valor` se conocía al inicio del periodo;
  (4) pocos años y pocos tickers: resultados ilustrativos, no concluyentes.

In [ ]:
import itertools

# --- Parámetros del backtest (se pueden cambiar y volver a correr) ---
# De dónde salen los tickers/métodos elegidos cada año:
#   'cambio_hits'  -> los de la tabla `cambio_hits` (hits de los métodos al 100% en 5 años)
#   'walk_forward' -> los que pasaban el filtro de hits usando solo lo que se sabía al decidir
FUENTE_CANDIDATOS = 'cambio_hits'
CAPITAL = 20_000                   # USD que se invierten cada año
HOLD_MESES = 12                    # horizonte fijo de cada compra
VENTANA_BT = 5                     # (walk_forward) años hacia atrás para evaluar el filtro de hits
MINIMO_PERIODOS_BT = 3             # (walk_forward) periodos ya resueltos mínimos
SEMANAS_CONFIRMACION = 156         # (walk_forward) un hit=0 se confirma como fallo tras 156 semanas
N_POSICIONES = [5, 10, None]       # None = todos los candidatos del año
COMPRA_MESES_DESPUES = [0, 3, 6]   # compra N meses después del inicio del periodo (0 = en el mes)
PESOS = ['igual', 'consenso']      # consenso = más peso a tickers con más métodos
N_SIMULACIONES = 500               # carteras al azar por año y combinación
SEMILLA = 0

CARPETA_SP500 = "Datos históricos del S&P 500 TR.csv"

# Precios: primera semana de cada mes (apertura), por ticker -> Serie indexada por Mes.
PX = {}
for t in sorted(hits['Ticket'].unique()):
    p = precio_inicio_por_mes(t)
    PX[t] = p.set_index('Mes')['Precio_inicio']

# Benchmark: S&P 500 Total Return (incluye dividendos, como los precios ajustados de los CSV).
_sp = pd.read_csv(CARPETA_SP500, encoding='utf-8-sig', thousands=',')
_sp['Fecha'] = pd.to_datetime(_sp['Fecha'], format='%d.%m.%Y')
_sp = _sp.sort_values('Fecha')
_sp['Mes'] = _sp['Fecha'].dt.to_period('M')
SP500 = _sp.groupby('Mes').first()['Apertura']
print(f"{len(PX)} tickers con precios; S&P 500 TR desde {_sp['Fecha'].min().date()} hasta {_sp['Fecha'].max().date()}")

In [ ]:
def metodo_pasa_en(historial, fecha_decision, ventana=VENTANA_BT, minimo=MINIMO_PERIODOS_BT):
    """¿El método (historial = sus periodos) habría estado al 100% en `fecha_decision`, usando
    SOLO lo que se sabía en ese momento? Cuenta periodos de la ventana ya resueltos:
      - acierto: hit=1 y su hit_date ya ocurrió,
      - fallo confirmado: hit=0 y ya pasaron 156 semanas desde su inicio.
    Los periodos aún pendientes no cuentan. Pasa si hay >= `minimo` resueltos y 0 fallos."""
    w = historial[(historial['Fecha_ini'] < fecha_decision) &
                  (historial['Fecha_ini'] >= fecha_decision - pd.DateOffset(years=ventana))]
    aciertos = ((w['hit'] == 1) & (w['hit_date'] <= fecha_decision)).sum()
    fallos = ((w['hit'] == 0) &
              (w['Fecha_ini'] + pd.Timedelta(weeks=SEMANAS_CONFIRMACION) <= fecha_decision)).sum()
    return (aciertos + fallos) >= minimo and fallos == 0


# Todos los periodos con su precio de inicio de mes (apertura de la 1a semana del mes).
base_hits = hits.copy()
base_hits['hit_date'] = pd.to_datetime(base_hits['hit_date'])
base_hits['Mes'] = pd.to_datetime(base_hits['Año'], format='%b%Y').dt.to_period('M')
base_hits = base_hits.rename(columns={'Ticket': 'Ticker'})
base_hits = base_hits.merge(
    pd.concat([precio_inicio_por_mes(t) for t in sorted(base_hits['Ticker'].unique())])
    .rename(columns={'Ticket': 'Ticker', 'Fecha_inicio': 'Fecha_ini', 'Precio_inicio': 'P_ini'}),
    on=['Ticker', 'Mes'], how='left')

# Candidatos walk-forward: (ticker, método, periodo) que pasaban el filtro EN ESE MOMENTO.
# Se evalúan todos los periodos con Valor, hayan o no pegado después (eso es futuro).
filas = []
for (ticker, metodo), hist in base_hits.groupby(['Ticker', 'Metodo']):
    for _, p in hist[hist['Valor'].notna() & hist['P_ini'].notna()].iterrows():
        if metodo_pasa_en(hist, p['Fecha_ini']):
            filas.append((ticker, metodo, p['Mes'], p['Fecha_ini'], p['P_ini'], p['Valor']))
candidatos_walk_forward = pd.DataFrame(
    filas, columns=['Ticker', 'Metodo', 'Mes', 'Fecha_ini', 'P_ini', 'Valor'])
candidatos_walk_forward['Cambio_esperado'] = (
    candidatos_walk_forward['Valor'] / candidatos_walk_forward['P_ini'] - 1)
candidatos_walk_forward['Anio'] = candidatos_walk_forward['Mes'].dt.year

# Candidatos de la tabla `cambio_hits`: cada fila es un (ticker, método, periodo) de un método
# al 100% en los últimos 5 años que SÍ llegó a su objetivo (hit=1). Su Cambio_% (Valor vs precio
# de inicio de mes) es el cambio esperado con el que se ordena y se eligen los tickers del año.
candidatos_cambio_hits = pd.DataFrame({
    'Ticker': cambio_hits['Ticket'],
    'Metodo': cambio_hits['Metodo'],
    'Mes': pd.to_datetime(cambio_hits['Año'], format='%b%Y').dt.to_period('M'),
    'Fecha_ini': cambio_hits['Fecha_inicio'],
    'P_ini': cambio_hits['Precio_inicio'],
    'Valor': cambio_hits['Valor'],
    'Cambio_esperado': cambio_hits['Cambio_%'] / 100,
})
candidatos_cambio_hits['Anio'] = candidatos_cambio_hits['Mes'].dt.year

candidatos = {'cambio_hits': candidatos_cambio_hits,
              'walk_forward': candidatos_walk_forward}[FUENTE_CANDIDATOS]

print(f"Fuente de candidatos: {FUENTE_CANDIDATOS}")
print(f"{len(candidatos)} (ticker, método, periodo) candidatos")
print("Tickers candidatos por año:", candidatos.groupby('Anio')['Ticker'].nunique().to_dict())

In [ ]:
def retorno_12m(tabla, mes, meses_despues):
    """Retorno de comprar en la apertura de la 1a semana de `mes + meses_despues` y vender
    HOLD_MESES después (misma referencia). NaN si no hay datos de salida."""
    entrada, salida = mes + meses_despues, mes + meses_despues + HOLD_MESES
    if entrada in tabla.index and salida in tabla.index:
        return tabla[salida] / tabla[entrada] - 1
    return np.nan


def armar_cartera(cand_anio, n_posiciones, peso):
    """Un renglón por ticker. Solo compra (Cambio_esperado > 0). Se ordena por la mediana del
    Cambio_esperado de sus métodos y se toman los `n_posiciones` mejores (None = todos).
    peso='igual' -> mismo peso; peso='consenso' -> proporcional al número de métodos que lo respaldan."""
    c = cand_anio[cand_anio['Cambio_esperado'] > 0]
    cart = (c.groupby('Ticker')
             .agg(Cambio_esperado=('Cambio_esperado', 'median'),
                  N_metodos=('Metodo', 'size'), Mes=('Mes', 'min'))
             .sort_values('Cambio_esperado', ascending=False))
    if n_posiciones:
        cart = cart.head(n_posiciones)
    if cart.empty:
        return cart
    cart['Cambio_esperado_%'] = cart['Cambio_esperado'] * 100
    cart['Peso'] = cart['N_metodos'] / cart['N_metodos'].sum() if peso == 'consenso' else 1 / len(cart)
    return cart


def backtest(candidatos, n_list, offsets, pesos, n_sim=N_SIMULACIONES, semilla=SEMILLA):
    """Backtest por año de inversión (cada año se invierten CAPITAL USD), comparado con el
    S&P 500 TR (mismas fechas y pesos) y con carteras al azar (mismo # de acciones, peso igual,
    mismas fechas, sacadas de los tickers que tenían periodo ese año)."""
    rng = np.random.default_rng(semilla)
    filas = []
    universo_anio = base_hits.groupby([base_hits['Mes'].dt.year, 'Ticker'])['Mes'].min()
    for n_pos, off, peso in itertools.product(n_list, offsets, pesos):
        for anio in sorted(candidatos['Anio'].unique()):
            cart = armar_cartera(candidatos[candidatos['Anio'] == anio], n_pos, peso)
            if cart.empty:
                continue
            cart['Ret'] = [retorno_12m(PX[t], m, off) for t, m in zip(cart.index, cart['Mes'])]
            cart['Ret_SP'] = [retorno_12m(SP500, m, off) for m in cart['Mes']]
            cart = cart.dropna(subset=['Ret'])
            if cart.empty:
                continue
            w = cart['Peso'] / cart['Peso'].sum()
            ret_cartera = float((w * cart['Ret']).sum())
            uni = universo_anio.loc[anio]
            r_uni = pd.Series({t: retorno_12m(PX[t], m, off) for t, m in uni.items()}).dropna()
            k = min(len(cart), len(r_uni))
            sims = np.array([r_uni.sample(k, random_state=int(rng.integers(1_000_000_000))).mean()
                             for _ in range(n_sim)])
            filas.append({
                'N': n_pos or 'todos', 'Compra_meses_despues': off, 'Peso': peso, 'Anio': anio,
                'Posiciones': len(cart), 'Retorno': ret_cartera,
                'Retorno_SP500': float((w * cart['Ret_SP']).sum()),
                'Retorno_azar': float(sims.mean()),
                'Pct_azar_que_supera': float((sims > ret_cartera).mean()),
            })
    return pd.DataFrame(filas)


resultados_bt = backtest(candidatos, N_POSICIONES, COMPRA_MESES_DESPUES, PESOS)
print(f"{len(resultados_bt)} filas (combinación x año)")

In [ ]:
def resumen_backtest(res):
    """Una fila por combinación (N, Compra_meses_despues, Peso) con métricas de todos los años."""
    r = res.assign(
        Gana_SP=res['Retorno'] > res['Retorno_SP500'],
        Gana_azar=res['Retorno'] > res['Retorno_azar'],
        Ganancia_USD=res['Retorno'] * CAPITAL,
        Ganancia_SP_USD=res['Retorno_SP500'] * CAPITAL,
        Ganancia_azar_USD=res['Retorno_azar'] * CAPITAL,
    )
    out = r.groupby(['N', 'Compra_meses_despues', 'Peso']).agg(
        Anios=('Anio', 'size'),
        Ret_prom=('Retorno', 'mean'),
        Ret_mediana=('Retorno', 'median'),
        Ret_SP500=('Retorno_SP500', 'mean'),
        Ret_azar=('Retorno_azar', 'mean'),
        Pct_anios_gana_SP=('Gana_SP', 'mean'),
        Pct_anios_gana_azar=('Gana_azar', 'mean'),
        Peor_anio=('Retorno', 'min'),
        Ganancia_USD=('Ganancia_USD', 'sum'),
        Ganancia_SP_USD=('Ganancia_SP_USD', 'sum'),
        Ganancia_azar_USD=('Ganancia_azar_USD', 'sum'),
    ).reset_index()
    pct = ['Ret_prom', 'Ret_mediana', 'Ret_SP500', 'Ret_azar', 'Peor_anio']
    out[pct] = (out[pct] * 100).round(1)
    out['Pct_anios_gana_SP'] = (out['Pct_anios_gana_SP'] * 100).round(0)
    out['Pct_anios_gana_azar'] = (out['Pct_anios_gana_azar'] * 100).round(0)
    usd = ['Ganancia_USD', 'Ganancia_SP_USD', 'Ganancia_azar_USD']
    out[usd] = out[usd].round(0)
    return out


def detalle_cartera(anio, n_posiciones, compra_meses_despues, peso):
    """Posiciones de la cartera de un año: ticker, #métodos, peso, USD invertidos, fecha de compra
    y retorno real a 12 meses."""
    cand = candidatos[candidatos['Anio'] == anio]
    cart = armar_cartera(cand, n_posiciones, peso)
    cart['Entrada'] = [(m + compra_meses_despues).to_timestamp() for m in cart['Mes']]
    cart['Retorno_12m_%'] = [retorno_12m(PX[t], m, compra_meses_despues) * 100
                             for t, m in zip(cart.index, cart['Mes'])]
    cart['USD_invertido'] = cart['Peso'] * CAPITAL
    return cart[['N_metodos', 'Cambio_esperado_%', 'Peso', 'USD_invertido', 'Entrada',
                 'Retorno_12m_%']].round(2)


tabla_backtest = resumen_backtest(resultados_bt)
print(f"Capital por año: {CAPITAL:,} USD. Retornos a {HOLD_MESES} meses, en %. "
      f"Ganancia_*_USD = suma de las ganancias de cada año (cada año invierte {CAPITAL:,} USD).")
display(tabla_backtest.sort_values('Ret_prom', ascending=False))

tabla_backtest.to_excel("Backtest_resumen.xlsx", index=False)
resultados_bt.to_excel("Backtest_por_anio.xlsx", index=False)
print("Guardados Backtest_resumen.xlsx y Backtest_por_anio.xlsx")

## Rehacer la cartera año a año

Simula a una persona que cada año rehace su cartera con los tickers elegidos ese año
(`FUENTE_CANDIDATOS`: por defecto los de `cambio_hits`, ver la nota de look-ahead arriba) y
reinvierte lo que le dejó el año anterior, empezando con 20,000 USD en el primer año con al menos
5 candidatos. Se muestra qué compra cada año, cuánto invierte en cada acción y cómo le fue a
12 meses, contra el S&P 500 TR y contra el promedio de carteras al azar. Se puede cambiar la
configuración en `CONFIG_REBAL` (N de posiciones, meses de espera para comprar, tipo de peso).

In [ ]:
import matplotlib.pyplot as plt

# --- Rebalanceo año a año ---
# Primer año con al menos 5 tickers candidatos (y no antes de 2019); último año 2024 para que
# todas las compras tengan ya 12 meses de datos.
_tickers_por_anio = candidatos.groupby('Anio')['Ticker'].nunique()
ANIO_INI_REBAL = max(2019, int(_tickers_por_anio[_tickers_por_anio >= 5].index.min()))
ANIO_FIN_REBAL = 2024
CONFIG_REBAL = dict(n_posiciones=5, compra_meses_despues=0, peso='igual')


def posiciones_del_anio(anio, n_posiciones, compra_meses_despues, peso, capital):
    """Cartera del año `anio` (candidatos según FUENTE_CANDIDATOS): una fila por ticker con
    fechas y precios de compra/venta (12 meses), USD invertidos y USD finales."""
    cart = armar_cartera(candidatos[candidatos['Anio'] == anio], n_posiciones, peso)
    filas = []
    for ticker, r in cart.iterrows():
        entrada = r['Mes'] + compra_meses_despues
        salida = entrada + HOLD_MESES
        px = PX[ticker]
        if entrada not in px.index or salida not in px.index:
            continue
        filas.append({
            'Anio': anio, 'Ticker': ticker, 'N_metodos': int(r['N_metodos']),
            'Cambio_esperado_%': r['Cambio_esperado_%'],
            'Compra': str(entrada), 'Venta': str(salida),
            'Precio_compra': px[entrada], 'Precio_venta': px[salida],
            'Peso': r['Peso'],
        })
    pos = pd.DataFrame(filas)
    if pos.empty:
        return pos
    pos['Peso'] = pos['Peso'] / pos['Peso'].sum()
    pos['USD_invertido'] = pos['Peso'] * capital
    pos['Retorno_%'] = (pos['Precio_venta'] / pos['Precio_compra'] - 1) * 100
    pos['USD_final'] = pos['USD_invertido'] * (1 + pos['Retorno_%'] / 100)
    return pos


def rebalanceo_anual(n_posiciones, compra_meses_despues, peso,
                     anio_ini=ANIO_INI_REBAL, anio_fin=ANIO_FIN_REBAL, capital_inicial=CAPITAL):
    """Cada año se rehace la cartera con el capital que dejó el año anterior (reinversión
    secuencial: el resultado del año Y se reinvierte en el año Y+1; es una aproximación porque
    cada ticker compra en su propio mes). Compara con el S&P 500 TR (mismas fechas y pesos) y
    con el promedio de carteras al azar de `resultados_bt`."""
    cap, cap_sp, cap_az = capital_inicial, capital_inicial, capital_inicial
    posiciones, resumen = [], []
    for anio in range(anio_ini, anio_fin + 1):
        pos = posiciones_del_anio(anio, n_posiciones, compra_meses_despues, peso, cap)
        if pos.empty:
            # Sin candidatos ese año: el capital se queda quieto.
            resumen.append({'Anio': anio, 'Posiciones': 0, 'Tickers': '(sin candidatos)',
                            'Capital_inicial': cap, 'Capital_final': cap,
                            'Capital_SP500': cap_sp, 'Capital_azar': cap_az})
            continue
        ret = pos['USD_final'].sum() / pos['USD_invertido'].sum() - 1
        ret_sp = sum(
            w * (SP500[pd.Period(v)] / SP500[pd.Period(c)] - 1)
            for w, c, v in zip(pos['Peso'], pos['Compra'], pos['Venta'])
        )
        fila_az = resultados_bt[(resultados_bt['N'] == (n_posiciones or 'todos')) &
                                (resultados_bt['Compra_meses_despues'] == compra_meses_despues) &
                                (resultados_bt['Peso'] == peso) & (resultados_bt['Anio'] == anio)]
        ret_az = float(fila_az['Retorno_azar'].iloc[0]) if len(fila_az) else np.nan
        resumen.append({
            'Anio': anio, 'Posiciones': len(pos), 'Tickers': ', '.join(pos['Ticker']),
            'Capital_inicial': cap, 'Capital_final': cap * (1 + ret), 'Retorno_%': ret * 100,
            'Retorno_SP500_%': ret_sp * 100, 'Retorno_azar_%': ret_az * 100,
        })
        cap *= 1 + ret
        cap_sp *= 1 + ret_sp
        cap_az *= 1 + (0 if np.isnan(ret_az) else ret_az)
        resumen[-1].update({'Capital_SP500': cap_sp, 'Capital_azar': cap_az})
        posiciones.append(pos)
    return (pd.concat(posiciones, ignore_index=True) if posiciones else pd.DataFrame()), pd.DataFrame(resumen)


pos_rebal, res_rebal = rebalanceo_anual(**CONFIG_REBAL)

print(f"Fuente: {FUENTE_CANDIDATOS}. Configuración: {CONFIG_REBAL}. Capital inicial {CAPITAL:,} USD "
      f"en {ANIO_INI_REBAL}, se reinvierte cada año ({ANIO_INI_REBAL}-{ANIO_FIN_REBAL}).\n")
for anio, grupo in pos_rebal.groupby('Anio'):
    print(f"=== {anio}: cartera elegida ===")
    display(grupo[['Ticker', 'N_metodos', 'Cambio_esperado_%', 'Compra', 'Venta', 'Peso',
                   'USD_invertido', 'USD_final', 'Retorno_%']].round(2).reset_index(drop=True))

print("Resumen año a año:")
display(res_rebal.round(1))
print(f"Capital final: cartera {res_rebal['Capital_final'].iloc[-1]:,.0f} USD | "
      f"S&P 500 TR {res_rebal['Capital_SP500'].iloc[-1]:,.0f} USD | "
      f"azar {res_rebal['Capital_azar'].iloc[-1]:,.0f} USD")

# Evolución del capital
x = [str(ANIO_INI_REBAL - 1)] + res_rebal['Anio'].astype(str).tolist()
fig, ax = plt.subplots(figsize=(8, 4))
for col, etiqueta in [('Capital_final', 'Cartera'), ('Capital_SP500', 'S&P 500 TR'), ('Capital_azar', 'Azar')]:
    ax.plot(x, [CAPITAL] + res_rebal[col].tolist(), marker='o', label=etiqueta)
ax.set_title('Capital (USD) rehaciendo la cartera cada año')
ax.set_xlabel('Fin del año')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

res_rebal.to_excel("Backtest_rebalanceo_anual.xlsx", index=False)
pos_rebal.to_excel("Backtest_rebalanceo_posiciones.xlsx", index=False)

## Cambio esperado por método y formas de resumirlo

Para cada año del rebalanceo muestra:
1. **Cambio esperado (%) de cada método** para cada ticker (`Valor / precio de inicio − 1`). Los negativos
   se muestran aunque `armar_cartera` los descarte.
2. **Distintas formas de resumirlo por ticker:** mínimo, mediana de todos, mediana de positivos (la que se
   usa hoy), promedio de positivos y máximo, junto con el número de métodos y el retorno real a 12 meses.
3. **Comparación de criterios:** cómo habría rendido la cartera si los tickers se ordenaran con cada
   criterio, en vez de con la mediana de positivos. Sirve para decidir el siguiente paso, pero con
   pocos años y con la fuente `cambio_hits` (que usa información futura) no es una conclusión firme.

`SOLO_TICKERS_DE_LA_CARTERA = False` muestra todos los candidatos del año, no solo los que compró la cartera.

In [ ]:
SOLO_TICKERS_DE_LA_CARTERA = True   # True: solo los tickers que compró la cartera; False: todos los candidatos del año
N_COMPARAR = 5                       # posiciones para comparar criterios de selección
SOLO_COMPRA = True                   # en la comparación de criterios, exige criterio > 0


def cambios_por_metodo(anio, tickers=None):
    """Cambio esperado (%) de cada método para cada ticker en `anio`: un renglón por ticker y
    una columna por método (vacío = ese método no era candidato ese año). Incluye los negativos."""
    c = candidatos[candidatos['Anio'] == anio]
    if tickers is not None:
        c = c[c['Ticker'].isin(tickers)]
    tabla = c.pivot_table(index='Ticker', columns='Metodo', values='Cambio_esperado', aggfunc='median') * 100
    return tabla.round(1)


def criterios_por_ticker(anio, tickers=None):
    """Distintas formas de resumir los cambios esperados de los métodos de cada ticker en `anio`.
    'Mediana_positivos_%' es la que usa hoy `armar_cartera`."""
    c = candidatos[candidatos['Anio'] == anio]
    if tickers is not None:
        c = c[c['Ticker'].isin(tickers)]
    todos = c.groupby('Ticker')['Cambio_esperado']
    pos = c[c['Cambio_esperado'] > 0].groupby('Ticker')['Cambio_esperado']
    out = pd.DataFrame({
        'N_metodos': todos.size(),
        'N_positivos': pos.size(),
        'Minimo_%': todos.min() * 100,
        'Mediana_todos_%': todos.median() * 100,
        'Mediana_positivos_%': pos.median() * 100,
        'Promedio_positivos_%': pos.mean() * 100,
        'Maximo_%': todos.max() * 100,
    })
    out['N_positivos'] = out['N_positivos'].fillna(0).astype(int)
    return out.sort_values('Mediana_positivos_%', ascending=False).round(1)


# Cada criterio recibe los cambios esperados de TODOS los métodos del ticker (Serie) y devuelve un número.
def _pos(s):
    return s[s > 0]

CRITERIOS = {
    'Mediana de positivos (actual)': lambda s: _pos(s).median(),
    'Promedio de positivos':         lambda s: _pos(s).mean(),
    'Mediana de todos':              lambda s: s.median(),
    'Mínimo (conservador)':          lambda s: s.min(),
    'Máximo (optimista)':            lambda s: s.max(),
    'Mediana de positivos, >=2 métodos': lambda s: _pos(s).median() if len(_pos(s)) >= 2 else np.nan,
}


def retorno_con_criterio(anio, criterio, n_posiciones=N_COMPARAR, compra_meses_despues=0):
    """Retorno de la cartera de `anio` (peso igual) si los tickers se ordenaran con `criterio`."""
    c = candidatos[candidatos['Anio'] == anio]
    valor = c.groupby('Ticker')['Cambio_esperado'].apply(criterio).dropna()
    if SOLO_COMPRA:
        valor = valor[valor > 0]
    elegidos = valor.sort_values(ascending=False).head(n_posiciones).index
    meses = c.groupby('Ticker')['Mes'].min()
    rets = pd.Series({t: retorno_12m(PX[t], meses[t], compra_meses_despues) for t in elegidos}).dropna()
    return (rets.mean(), list(rets.index)) if len(rets) else (np.nan, [])


# --- 1) Cambios esperados por método, año por año ---
for anio in range(ANIO_INI_REBAL, ANIO_FIN_REBAL + 1):
    tickers = (pos_rebal.loc[pos_rebal['Anio'] == anio, 'Ticker'].tolist()
               if SOLO_TICKERS_DE_LA_CARTERA else None)
    print(f"=== {anio}: cambio esperado (%) por método ===")
    display(cambios_por_metodo(anio, tickers))
    print(f"=== {anio}: distintas formas de resumirlo por ticker ===")
    crit = criterios_por_ticker(anio, tickers)
    # Retorno real a 12 meses de lo que compró la cartera (si el ticker estuvo en ella).
    real = pos_rebal[pos_rebal['Anio'] == anio].set_index('Ticker')['Retorno_%'].round(1)
    display(crit.join(real.rename('Retorno_real_12m_%')))

# --- 2) ¿Qué criterio de selección habría rendido más? (N_COMPARAR posiciones, peso igual) ---
anios_rebal = list(range(ANIO_INI_REBAL, ANIO_FIN_REBAL + 1))
comparacion = pd.DataFrame(
    {nombre: {anio: retorno_con_criterio(anio, f)[0] * 100 for anio in anios_rebal}
     for nombre, f in CRITERIOS.items()}
).round(1)
comparacion.index.name = 'Anio'
capital_final = [round(CAPITAL * np.prod(1 + comparacion.loc[anios_rebal, col].dropna() / 100))
                 for col in comparacion.columns]
comparacion.loc['Promedio'] = comparacion.loc[anios_rebal].mean().round(1)
comparacion.loc['Capital final USD'] = capital_final
print(f"Retorno a 12 meses (%) de una cartera de {N_COMPARAR} tickers según el criterio de ordenamiento, "
      f"{ANIO_INI_REBAL}-{ANIO_FIN_REBAL}. Capital final = {CAPITAL:,} USD reinvertidos cada año.")
display(comparacion)

with pd.ExcelWriter("Cambios_esperados_por_metodo.xlsx") as xl:
    for anio in anios_rebal:
        cambios_por_metodo(anio).to_excel(xl, sheet_name=str(anio))
    comparacion.to_excel(xl, sheet_name='Comparacion_criterios')
print("Guardado Cambios_esperados_por_metodo.xlsx (una hoja por año, todos los candidatos, y la comparación)")